In [2]:
import pandas as pd
import numpy as np

# Dicionário com os arquivos estratégicos 
arquivos = {
    "Ativo e Passivo (Mensal)": "inf_mensal_fii_ativo_passivo_2025.csv",
    "Complemento (Mensal)": "inf_mensal_fii_complemento_2025.csv",
    "Geral (Mensal)": "inf_mensal_fii_geral_2025.csv",
    "Desempenho Imóvel (Trimestral)": "inf_trimestral_fii_imovel_desempenho_2025.csv",
    "Base trimestral":"inf_trimestral_fii_imovel_2025.csv"
}


In [3]:

# 1. Carregar apenas as colunas que queremos
colunas_passivo = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Total_Passivo']
colunas_comp = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Valor_Ativo', 'Patrimonio_Liquido', 
                'Valor_Patrimonial_Cotas', 'Percentual_Dividend_Yield_Mes']
#1.1 Carregando os csvs
df_passivo = pd.read_csv('inf_mensal_fii_ativo_passivo_2025.csv', sep=';', encoding='latin1', usecols=colunas_passivo)
df_comp = pd.read_csv('inf_mensal_fii_complemento_2025.csv', sep=';', encoding='latin1', usecols=colunas_comp)

# 2. Converter datas e isolar para o mês mais recente (último mês disponível)
df_passivo['Data_Referencia'] = pd.to_datetime(df_passivo['Data_Referencia'])
df_comp['Data_Referencia'] = pd.to_datetime(df_comp['Data_Referencia'])
ultima_data = df_comp['Data_Referencia'].max()
df_passivo_atual = df_passivo[df_passivo['Data_Referencia'] == ultima_data]
df_comp_atual = df_comp[df_comp['Data_Referencia'] == ultima_data]

# 3. O Cruzamento (Inner Join usando CNPJ e Data)
df_micro = pd.merge(df_comp_atual, df_passivo_atual, on=['CNPJ_Fundo_Classe', 'Data_Referencia'], how='inner')

# 4. Cálculo do LTV e padronização dos nomes para o modelo
df_micro['LTV'] = df_micro['Total_Passivo'] / df_micro['Valor_Ativo']
df_micro.rename(columns={
    'Valor_Patrimonial_Cotas': 'VPA', 
    'Percentual_Dividend_Yield_Mes': 'DY_Mes'
}, inplace=True)

# Visualizando a Matriz de Fundamentos limpa
print("\n--- Base de Fundamentos Extraída ---")
print(df_micro[['CNPJ_Fundo_Classe', 'LTV', 'VPA', 'DY_Mes']].head())


--- Base de Fundamentos Extraída ---
    CNPJ_Fundo_Classe       LTV          VPA    DY_Mes
0  00.332.266/0001-31  0.001643    92.127221  0.000000
1  00.613.094/0001-74  1.127006   -32.997029 -0.000908
2  00.762.723/0001-28  0.016373    17.800085  0.000000
3  00.868.235/0001-08  0.020158  2239.480309  0.005884
4  01.201.140/0001-90  0.007323   110.796096  0.005895


In [4]:


# 1. CARREGAR E ISOLAR DADOS FÍSICOS (Trimestral)
colunas_imovel = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Percentual_Vacancia', 
                  'Percentual_Inadimplencia', 'Area']

# Lendo o CSV com as colunas corretas
df_imoveis = pd.read_csv('inf_trimestral_fii_imovel_2025.csv', sep=';', encoding='latin1', usecols=colunas_imovel)

# Isolando a "foto" mais recente do trimestre
df_imoveis['Data_Referencia'] = pd.to_datetime(df_imoveis['Data_Referencia'])
ultima_data_tri = df_imoveis['Data_Referencia'].max()
df_imoveis_atual = df_imoveis[df_imoveis['Data_Referencia'] == ultima_data_tri].copy()


# 2. TRATAMENTO E AGREGAÇÃO (Média Ponderada)
# Garantir que as colunas numéricas estão no formato certo e remover nulos (ex: FIIs de Papel)
df_imoveis_atual['Area'] = pd.to_numeric(df_imoveis_atual['Area'], errors='coerce')
df_imoveis_atual['Percentual_Vacancia'] = pd.to_numeric(df_imoveis_atual['Percentual_Vacancia'], errors='coerce')
df_imoveis_atual['Percentual_Inadimplencia'] = pd.to_numeric(df_imoveis_atual['Percentual_Inadimplencia'], errors='coerce')

# Filtramos os imóveis que possuem os dados preenchidos
df_imoveis_validos = df_imoveis_atual.dropna(subset=['Area', 'Percentual_Vacancia']).copy()

# Criar a coluna do valor absoluto da vacância (Vacância * Peso)
df_imoveis_validos['Vacancia_Area_Absoluta'] = df_imoveis_validos['Percentual_Vacancia'] * df_imoveis_validos['Area']

# Agrupar somando as áreas totais e as vacâncias absolutas por Fundo
df_agregado = df_imoveis_validos.groupby('CNPJ_Fundo_Classe').agg({
    'Vacancia_Area_Absoluta': 'sum',
    'Area': 'sum',
    'Percentual_Inadimplencia': 'mean' # Inadimplência segue na média simples por enquanto
}).reset_index()

# Dividir a Vacância Absoluta pela Área Total para achar o Percentual Ponderado final
df_agregado['Percentual_Vacancia'] = df_agregado['Vacancia_Area_Absoluta'] / df_agregado['Area']

# Isolar as colunas limpas
df_vacancia_fundo = df_agregado[['CNPJ_Fundo_Classe', 'Percentual_Vacancia', 'Percentual_Inadimplencia']]


# 3. O MERGE (Unindo Contabilidade e Física)
# Assumindo que o df_micro (com LTV e VPA) já está na memória
df_matriz_quase_pronta = pd.merge(df_micro, df_vacancia_fundo, on='CNPJ_Fundo_Classe', how='left')

# Tratamento de Valores Nulos (O Ponto Cego da CVM)
# FIIs de Papel não reportam vacância física, então o merge vai gerar NaN.
# Precisamos preencher com zero para que o algoritmo de Machine Learning consiga calcular a distância.
df_matriz_quase_pronta['Percentual_Vacancia'] = df_matriz_quase_pronta['Percentual_Vacancia'].fillna(0)
df_matriz_quase_pronta['Percentual_Inadimplencia'] = df_matriz_quase_pronta['Percentual_Inadimplencia'].fillna(0)

print("\n--- Matriz Microeconômica Consolidada ---")
print(df_matriz_quase_pronta.head(70))


--- Matriz Microeconômica Consolidada ---
     CNPJ_Fundo_Classe Data_Referencia   Valor_Ativo  Patrimonio_Liquido  \
0   00.332.266/0001-31      2025-12-01  2.583944e+08        2.579699e+08   
1   00.613.094/0001-74      2025-12-01  1.970381e+08       -2.502495e+07   
2   00.762.723/0001-28      2025-12-01  9.744736e+07        9.585189e+07   
3   00.868.235/0001-08      2025-12-01  1.241421e+08        1.216396e+08   
4   01.201.140/0001-90      2025-12-01  5.255968e+08        5.217479e+08   
..                 ...             ...           ...                 ...   
65  11.769.604/0001-13      2025-12-01  1.922764e+09        1.304610e+09   
66  11.827.568/0001-05      2025-12-01  5.047303e+07        5.035478e+07   
67  11.839.593/0001-09      2025-12-01  6.363438e+09        5.497692e+09   
68  11.945.604/0001-27      2025-12-01  2.488960e+06        2.388743e+06   
69  12.005.956/0001-65      2025-12-01  5.044952e+09        4.602846e+09   

            VPA    DY_Mes  Total_Passivo    

In [5]:
#Varedura de dados
# 1. Checagem de Valores Nulos (NaNs)
print("--- 1. VALORES NULOS (NaNs) ---")
nulos = df_matriz_quase_pronta.isna().sum()
print(nulos[nulos > 0])
if nulos.sum() == 0:
    print(" Nenhum valor nulo encontrado. Merge perfeito!")

# 2. Resumo Estatístico (O radar de Outliers)
print("\n--- 2. ESTATÍSTICAS DESCRITIVAS (Distribuição) ---")
colunas_numericas = ['LTV', 'VPA', 'DY_Mes', 'Percentual_Vacancia', 'Percentual_Inadimplencia']
print(df_matriz_quase_pronta[colunas_numericas].describe().round(4))

# 3. Alarmes de Regras de Negócio ( números bizarros)
print("\n--- 3. ALARMES DE ANOMALIAS ---")

# A) Vacância impossível (> 100% ou < 0%)
vacancia_estranha = df_matriz_quase_pronta[
    (df_matriz_quase_pronta['Percentual_Vacancia'] > 1) | 
    (df_matriz_quase_pronta['Percentual_Vacancia'] < 0)
]
print(f"⚠️ Fundos com Vacância > 1 ou < 0: {len(vacancia_estranha)}")
if len(vacancia_estranha) > 0:
    print(vacancia_estranha[['CNPJ_Fundo_Classe', 'Percentual_Vacancia']].head())

# B) LTV Negativo (Passivo ou Ativo negativo, o que é um erro contábil)
ltv_negativo = df_matriz_quase_pronta[df_matriz_quase_pronta['LTV'] < 0]
print(f"⚠️ Fundos com LTV negativo: {len(ltv_negativo)}")
if len(ltv_negativo) > 0:
    print(ltv_negativo[['CNPJ_Fundo_Classe', 'LTV']].head())

# C) LTV > 1 (Alavancagem Extrema / PL Negativo)
# Isso é possível na realidade (vimos um no seu print anterior), mas é bom isolar para entender o risco.
ltv_estourado = df_matriz_quase_pronta[df_matriz_quase_pronta['LTV'] > 1]
print(f"⚠️ Fundos com LTV > 1 (Distress): {len(ltv_estourado)}")

# D) VPA Negativo ou Zerado
vpa_estranho = df_matriz_quase_pronta[df_matriz_quase_pronta['VPA'] <= 0]
print(f"⚠️ Fundos com VPA <= 0: {len(vpa_estranho)}")

--- 1. VALORES NULOS (NaNs) ---
DY_Mes    60
LTV        3
dtype: int64

--- 2. ESTATÍSTICAS DESCRITIVAS (Distribuição) ---
             LTV           VPA     DY_Mes  Percentual_Vacancia  \
count  1222.0000  1.225000e+03  1165.0000            1225.0000   
mean      1.8461  5.583532e+04     0.0104               0.0610   
std      41.8249  8.413971e+05     0.0806               0.2049   
min      -1.6530 -4.415749e+04    -0.8566               0.0000   
25%       0.0021  1.839600e+01     0.0000               0.0000   
50%       0.0132  1.005510e+02     0.0000               0.0000   
75%       0.1255  7.038593e+02     0.0074               0.0000   
max    1166.0246  2.536967e+07     1.3744               1.0000   

       Percentual_Inadimplencia  
count                 1225.0000  
mean                     0.0078  
std                      0.0677  
min                     -0.0592  
25%                      0.0000  
50%                      0.0000  
75%                      0.0000  
max       

In [6]:
#Limpeza
# 1. Cópia de segurança
df_limpo = df_matriz_quase_pronta.copy()

# 2. Tratamento de NaNs
# Se não tem DY no mês, assume-se 0 de distribuição
df_limpo['DY_Mes'] = df_limpo['DY_Mes'].fillna(0)
# Dropa os 3 fundos sem LTV (não dá para clusterizar estrutura de capital sem LTV)
df_limpo = df_limpo.dropna(subset=['LTV'])

# 3. Poda de LTV (Regra de Negócio: 0 <= LTV <= 1.5)
# LTV = 1.5 significa passivo 50% maior que o ativo (distress severo, mas possível).
df_limpo = df_limpo[(df_limpo['LTV'] >= 0) & (df_limpo['LTV'] <= 1.5)]

# 4. Poda de Dividend Yield (Regra de Negócio: 0 <= DY <= 0.05)
# Corta amortizações bizarras e ajustes contábeis negativos
df_limpo = df_limpo[(df_limpo['DY_Mes'] >= 0) & (df_limpo['DY_Mes'] <= 0.05)]

# 5. Poda de VPA (Regra de Negócio: VPA > 0 e limitando aberrações de milhões)
# Corta fundos com PL negativo e fundos de 1 cota institucional
df_limpo = df_limpo[(df_limpo['VPA'] > 0) & (df_limpo['VPA'] < 50000)]

# 6. Avaliação da Perda de Dados
fundos_antes = len(df_matriz_quase_pronta)
fundos_depois = len(df_limpo)
perda = fundos_antes - fundos_depois

print(f"✅ Limpeza concluída!")
print(f"Fundos iniciais: {fundos_antes}")
print(f"Fundos após filtro: {fundos_depois}")
print(f"Total de distorções e outliers removidos: {perda} fundos.")

# Verifica se sobrou algum erro
print("\nNovas estatísticas máximas após a limpeza:")
print(df_limpo[['LTV', 'DY_Mes', 'VPA']].max())

✅ Limpeza concluída!
Fundos iniciais: 1225
Fundos após filtro: 1114
Total de distorções e outliers removidos: 111 fundos.

Novas estatísticas máximas após a limpeza:
LTV           1.000000
DY_Mes        0.049678
VPA       43620.748901
dtype: float64


In [8]:
#Aqui é feito o cruzamento dos dados da base bruta, para conectar o ticket do fundo com a base via ISIN( que nada mais é do que um id do fundo)
# 1. CARREGAR A BASE GERAL DA CVM (Onde está o CNPJ e o ISIN)
df_geral_cvm = pd.read_csv('inf_mensal_fii_geral_2025.csv', sep=';', encoding='latin1')

# 2. EXTRAIR O TICKER DO ISIN NA CVM
# O ISIN brasileiro é BR + TICKER + TIPO (ex: BRADSHCTF005). 
# O Ticker são os 4 caracteres após o 'BR'.
def extrair_ticker_isin(isin):
    if pd.isna(isin) or len(str(isin)) < 6:
        return None
    return str(isin)[2:6].upper()

df_geral_cvm['Ticker_CVM'] = df_geral_cvm['Codigo_ISIN'].apply(extrair_ticker_isin)

# 3. CARREGAR A BASE DA B3 
df_b3 = pd.read_csv('fundosListados.csv', sep=';', encoding='latin1')
df_b3['Ticker_B3'] = df_b3['Fundo'].astype(str).str.strip().str.upper()

# 4. O MERGE PELO TICKER 
df_dicionario = pd.merge(
    df_geral_cvm[['CNPJ_Fundo_Classe', 'Ticker_CVM']].dropna(),
    df_b3[['Ticker_B3']],
    left_on='Ticker_CVM',
    right_on='Ticker_B3',
    how='inner'
).drop_duplicates(subset='Ticker_B3')

# 5. UNIR COM A MATRIZ DE FUNDAMENTOS LIMPA
df_final_investivel = pd.merge(
    df_limpo, 
    df_dicionario[['CNPJ_Fundo_Classe', 'Ticker_B3']], 
    on='CNPJ_Fundo_Classe', 
    how='inner'
)

# Ajuste final para o Yahoo Finance
df_final_investivel['Ticker_YF'] = df_final_investivel['Ticker_B3'] + '11.SA'

print(f" Amostra expandida para {len(df_final_investivel)} fundos.")
print(df_final_investivel[['Ticker_YF', 'LTV', 'Percentual_Vacancia']].head(10))

 Amostra expandida para 399 fundos.
   Ticker_YF       LTV  Percentual_Vacancia
0  FVPQ11.SA  0.001643             0.103933
1  CTXT11.SA  0.016373             1.000000
2  ABCP11.SA  0.007323             0.012190
3  FTCE11.SA  0.200288             0.361283
4  FMOF11.SA  0.046206             0.041700
5  RTEL11.SA  0.036910             0.236553
6  FPAB11.SA  0.034359             0.236117
7  SHPH11.SA  0.005005             0.005464
8  RCRB11.SA  0.114576             0.009509
9  HCRI11.SA  0.013911             0.000000


In [9]:
#Aqui eu quis dar uma olhada em todos os fundos do dataset, e depois aplciar mais uma filtragem.
# 1. Configurar o Pandas para mostrar tudo
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 2. Exibir a lista ordenada pelo Ticker para facilitar a conferência
print("--- LISTA COMPLETA DE FIIs INVESTÍVEIS (N=399) ---")
df_ordenado = df_final_investivel.sort_values('Ticker_YF')

# Exibe o Ticker, CNPJ e os fundamentos principais
print(df_ordenado[['Ticker_YF', 'CNPJ_Fundo_Classe', 'LTV', 'Percentual_Vacancia', 'VPA']])

# 3. Opcional: Salvar em Excel/CSV para a equipe analisar fora do Python
# df_ordenado.to_csv('matriz_micro_consolidada.csv', sep=';', index=False)
# print("\n✅ Arquivo 'matriz_micro_consolidada.csv' gerado para análise da equipe.")

--- LISTA COMPLETA DE FIIs INVESTÍVEIS (N=399) ---
     Ticker_YF   CNPJ_Fundo_Classe       LTV  Percentual_Vacancia           VPA
2    ABCP11.SA  01.201.140/0001-90  0.007323             0.012190    110.796096
394  ADSH11.SA  63.375.304/0001-53  0.490996             0.014000      9.700845
371  AERO11.SA  57.927.297/0001-52  0.215324             0.000000     99.729479
227  AFHI11.SA  36.642.293/0001-58  0.000934             0.000000     94.559785
209  AIEC11.SA  35.765.826/0001-26  0.022024             0.000000     76.573358
314  AJFI11.SA  51.472.985/0001-99  0.028331             0.066079     11.112882
15   ALMI11.SA  07.122.725/0001-00  0.024885             0.468000   2062.462425
143  ALZR11.SA  28.737.771/0001-85  0.250443             0.112501     10.640670
17   ANCR11.SA  07.789.135/0001-27  0.068858             0.042123    120.678243
268  APTO11.SA  41.081.356/0001-84  0.014313             0.000000      8.508690
288  APXM11.SA  43.010.485/0001-07  0.001352             1.000000    

In [10]:
#Aqui apliquei uam série de filtragens.

print("Análise de consistencia (N=399)")


# 1. Verificação de Escala de Cota (VPA vs Realidade)
# FIIs listados quase sempre valem entre R$ 5,00 e R$ 2.000,00.
# Valores fora disso indicam erros de base ou fundos extremamente ilíquidos.
vpa_suspeito = df_final_investivel[(df_final_investivel['VPA'] < 1) | (df_final_investivel['VPA'] > 5000)]
print(f" 1. Fundos com VPA fora da escala comercial (R$1 - R$5000): {len(vpa_suspeito)}")
if len(vpa_suspeito) > 0:
    print(vpa_suspeito[['Ticker_YF', 'VPA']].head(10))

# 2. Verificação de Conflito LTV vs Vacância
# Fundos de Papel (CRI) NÃO podem ter vacância física. 
# Se um fundo tem Vacância > 0 e é puramente de papel, há erro de classificação na CVM.
papel_com_vacancia = df_final_investivel[(df_final_investivel['Percentual_Vacancia'] > 0) & 
                                         (df_final_investivel['Ticker_YF'].str.contains('CR|CP|KN'))] # Tickers comuns de papel
print(f"2. Possíveis fundos de Papel com Vacância Física detectada: {len(papel_com_vacancia)}")

# 3. Análise de Outliers Multivariados (Z-Score)
# Vamos ver quem está a mais de 3 desvios padrão da média em LTV ou Vacância
from scipy import stats
cols_analise = ['LTV', 'Percentual_Vacancia']
z_scores = np.abs(stats.zscore(df_final_investivel[cols_analise]))
outliers_bolsa = df_final_investivel[(z_scores > 3).any(axis=1)]
print(f"3. Outliers Estatísticos detectados (Z-Score > 3): {len(outliers_bolsa)}")
if len(outliers_bolsa) > 0:
    print(outliers_bolsa[['Ticker_YF', 'LTV', 'Percentual_Vacancia']].head(10))

# 4. Verificação de Duplicidade de CNPJ
duplicados = df_final_investivel.duplicated(subset=['CNPJ_Fundo_Classe']).sum()
print(f"\n✅ 4. CNPJs duplicados na base: {duplicados}")

print("\n=========================================================")

Análise de consistencia (N=399)
 1. Fundos com VPA fora da escala comercial (R$1 - R$5000): 11
     Ticker_YF           VPA
62   KNRE11.SA      0.632653
97   ESTQ11.SA      0.014474
100  LOFT11.SA      0.121781
135  DLMT11.SA      0.515130
137  BPLC11.SA   9977.520587
254  SNLG11.SA      0.383496
299  AROA11.SA      0.961006
317  KLOG11.SA      0.947811
369  KPMR11.SA  10308.907804
372  KPRP11.SA      0.985339
2. Possíveis fundos de Papel com Vacância Física detectada: 9
3. Outliers Estatísticos detectados (Z-Score > 3): 28
     Ticker_YF       LTV  Percentual_Vacancia
1    CTXT11.SA  0.016373             1.000000
13   FAMB11.SA  0.183583             0.980100
38   PRSV11.SA  0.005934             1.000000
54   RBOP11.SA  0.035304             0.719200
82   TSNC11.SA  0.009993             1.000000
88   SAIC11.SA  0.029564             1.000000
91   FIVN11.SA  0.106260             0.853200
100  LOFT11.SA  0.362414             0.999382
118  BRIM11.SA  0.664623             0.000000
136  RECR1

In [11]:
#Aplicando o filtro
# 1. Poda de VPA (Removendo os 11 fundos de centavos/milhares)
df_micro_final= df_final_investivel[
    (df_final_investivel['VPA'] >= 1) & 
    (df_final_investivel['VPA'] <= 5000)
].copy()

# 2. Poda de Vacância Extrema (Opcional, mas recomendado para clareza dos clusters)
# Vamos remover quem tem vacância de 100%, pois são casos de falência operacional
df_micro_final = df_micro_final[df_micro_final['Percentual_Vacancia'] < 0.95]

# 3. Resultado Final
fundos_finais = len(df_micro_final)
print(f"Sobraram {fundos_finais} fundos consistentes.")
print(df_micro_final.head())

#Exportando o csv
df_micro_final.to_csv('df_micro_final.csv', index=True)




Sobraram 377 fundos consistentes.
    CNPJ_Fundo_Classe Data_Referencia   Valor_Ativo  Patrimonio_Liquido          VPA    DY_Mes  Total_Passivo       LTV  Percentual_Vacancia  Percentual_Inadimplencia Ticker_B3  Ticker_YF
0  00.332.266/0001-31      2025-12-01  2.583944e+08        2.579699e+08    92.127221  0.000000   4.244746e+05  0.001643             0.103933                  0.106500      FVPQ  FVPQ11.SA
2  01.201.140/0001-90      2025-12-01  5.255968e+08        5.217479e+08   110.796096  0.005895   3.848859e+06  0.007323             0.012190                  0.006539      ABCP  ABCP11.SA
3  01.235.622/0001-61      2025-12-01  4.758225e+09        3.805210e+09  2872.008412  0.011298   9.530144e+08  0.200288             0.361283                  0.000000      FTCE  FTCE11.SA
4  01.633.741/0001-72      2025-12-01  2.461035e+07        2.347321e+07    46.206371  0.031096   1.137140e+06  0.046206             0.041700                  0.000000      FMOF  FMOF11.SA
5  01.657.856/0001-05     